# model_experiment_XGBoost

Same setup as LightGBM — same `WalmartPreprocessor` features, same sample weights, same folds — so the LightGBM/XGBoost comparison isolates the *algorithm*: level-wise vs leaf-wise growth, different categorical-split and L1-objective implementations.

MLflow experiment: **XGBoost_Training**. CPU is enough.

In [1]:
# Kaggle bootstrap — does nothing when running locally.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow xgboost scikit-learn")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.6/954.6 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 12.7 MB/s eta 0:00:00


Cloning into 'walmart-sales-forecasting'...


In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pandas as pd
import mlflow
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow
from src.preprocessing import WalmartPreprocessor, make_xyw, BLEND_HOLIDAY_WEEKS
from src.postprocess import apply_christmas_shift, blend_holiday_naive

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("XGBoost_Training")

2026/07/11 13:33:42 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.


MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: XGBoost_Training


## Run 1 — Cleaning (same decisions as LightGBM, shared preprocessor)

In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_params({
        "preprocessor": "shared WalmartPreprocessor (identical features to LightGBM)",
        "objective": "reg:absoluteerror (L1, matches WMAE)",
        "sample_weight": "1 + 4*IsHoliday",
        "categoricals": "native (enable_categorical + hist)",
        "nan_handling": "native",
    })

🏃 View run XGBoost_Cleaning at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/e07a83b04df1481ea5694999c2037e8e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5


## Run 2 — Cross-validation on the shared folds

In [4]:
def eval_fold(params, fold):
    tr, va = split_fold(train, fold)
    Xtr, ytr, wtr = make_xyw(tr)
    Xva, yva, _ = make_xyw(va)
    prep = WalmartPreprocessor(features_df=features, stores_df=stores)
    Xtr_t = prep.fit(Xtr, ytr).transform(Xtr)
    Xva_t = prep.transform(Xva)
    model = XGBRegressor(**params)
    model.fit(Xtr_t, ytr, sample_weight=wtr)
    rep = wmae_report(yva, model.predict(Xva_t), va["IsHoliday"])
    return rep, model, Xtr_t, Xva_t

PARAM_GRID = [
    dict(objective="reg:absoluteerror", n_estimators=800, learning_rate=0.05,
         max_depth=8, min_child_weight=30, colsample_bytree=0.8,
         tree_method="hist", enable_categorical=True, random_state=42),
    dict(objective="reg:absoluteerror", n_estimators=1500, learning_rate=0.05,
         max_depth=12, min_child_weight=5, colsample_bytree=0.8, subsample=0.9,
         tree_method="hist", enable_categorical=True, random_state=42),
    dict(objective="reg:quantileerror", quantile_alpha=0.5,
         n_estimators=1500, learning_rate=0.05,
         max_depth=10, min_child_weight=10, colsample_bytree=0.8,
         tree_method="hist", enable_categorical=True, random_state=42),
]

results = []
for i, params in enumerate(PARAM_GRID):
    with mlflow.start_run(run_name=f"XGBoost_CV_{i}"):
        mlflow.log_params({**{k: str(v) for k, v in params.items()}, "fold": 1})
        rep, model, _, _ = eval_fold(params, fold=1)
        mlflow.log_metrics(rep)
    results.append((i, rep["wmae"]))
    print(i, rep)
best_i = min(results, key=lambda r: r[1])[0]
print("best config:", best_i)

🏃 View run XGBoost_CV_0 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/5eb3dcd83da443e7a2f4848adbb3485a
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
0 {'wmae': 2428.8749851353678, 'mae_holiday': 3330.6937784050942, 'mae_nonholiday': 2047.9502250348683}
🏃 View run XGBoost_CV_1 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/b703bea25e054cee9984facb49c1ba36
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
1 {'wmae': 2378.902787049028, 'mae_holiday': 3445.5173410388893, 'mae_nonholiday': 1928.3689368670357}
🏃 View run XGBoost_CV_2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/6d18d3babf214e0999107669a28fed9b
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
2 {'wmae': 2377.0836330276256, 

In [5]:
# fold 2 sanity
with mlflow.start_run(run_name="XGBoost_CV_best_fold2"):
    rep2, _, _, _ = eval_fold(PARAM_GRID[best_i], fold=2)
    mlflow.log_params({**{k: str(v) for k, v in PARAM_GRID[best_i].items()}, "fold": 2})
    mlflow.log_metrics(rep2)
print(rep2)

🏃 View run XGBoost_CV_best_fold2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/6a210f80a4354a97a2fb93a86b3a494b
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
{'wmae': 1609.91834599907, 'mae_holiday': 1646.9897346712157, 'mae_nonholiday': 1599.8288113201897}


## Run 3 — Holiday blend (no Christmas; rationale in the LightGBM notebook)

In [6]:
tr, va = split_fold(train, 1)
_, yva, _ = make_xyw(va)
rep, model, _, Xva_t = eval_fold(PARAM_GRID[best_i], fold=1)
va_pred = va[["Store", "Dept", "Date"]].copy()
va_pred["pred"] = model.predict(Xva_t).astype(float)

blend_scores = {}
for w in (0.0, 0.5, 0.75):
    blended = blend_holiday_naive(va_pred, tr, weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    r = wmae_report(yva, blended["pred"], va["IsHoliday"])
    blend_scores[w] = r["wmae"]
    with mlflow.start_run(run_name=f"XGBoost_Blend_noXmas_w{w}"):
        mlflow.log_params({"blend_weight": w, "fold": 1})
        mlflow.log_metrics(r)
    print(f"w={w}: {r}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

🏃 View run XGBoost_Blend_noXmas_w0.0 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/54eb04e03c3243d3ba80ba869bd74fd9
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
w=0.0: {'wmae': 2377.0836330276256, 'mae_holiday': 3350.503001087961, 'mae_nonholiday': 1965.915067875913}
🏃 View run XGBoost_Blend_noXmas_w0.5 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/128aaa79a2fe42c3a5d4a9695f84f55c
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
w=0.5: {'wmae': 2332.8309490238585, 'mae_holiday': 3201.484478085714, 'mae_nonholiday': 1965.915067875913}
🏃 View run XGBoost_Blend_noXmas_w0.75 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/e2b86d1655dd48c2ba78e402186c9b94
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/

## Run 4 — Final: Pipeline on ALL train, register, submission

In [7]:
BEST = PARAM_GRID[best_i]
X, y, w = make_xyw(train)
pipe = Pipeline([
    ("prep", WalmartPreprocessor(features_df=features, stores_df=stores)),
    ("model", XGBRegressor(**BEST)),
])
pipe.fit(X, y, model__sample_weight=w)

with mlflow.start_run(run_name="XGBoost_Final"):
    mlflow.log_params({**{k: str(v) for k, v in BEST.items()}, "blend_weight": BLEND_W})
    mlflow.log_metric("fold1_wmae", min(r[1] for r in results))
    mlflow.sklearn.log_model(pipe, name="model",
                             serialization_format="cloudpickle",
                             registered_model_name="walmart-xgboost")

sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = pipe.predict(test[["Store", "Dept", "Date"]]).astype(float)
sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)
make_submission(sub, "pred", "submission_xgboost.csv")
print(f"wrote submission_xgboost.csv (blend w={BLEND_W})")

2026/07/11 14:03:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'walmart-xgboost'.
2026/07/11 14:03:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-xgboost, version 1
Created version '1' of model 'walmart-xgboost'.


🏃 View run XGBoost_Final at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5/runs/3bd94eb2c3974601b79c31078d41250e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/5
wrote submission_xgboost.csv (blend w=0.75)
